
# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.1.6.2.1.1.1.1
## Covariant \(Dg,Dq,DX\) External-Connection Jet Correction

**Auteur :** Charlemagne O Laurince  
**Prédécesseur direct exécuté :** `.1.6.2.1.1.1`  
**Mission exclusive :** corriger les termes de connexion externe manquants dans les jets covariants avant le second jet métrique.

---

# État gelé à l'entrée

\[
\boxed{
R_{HH}^{\rm can}=0
\qquad
\texttt{STRONG\_ZERO}
}
\]

reste fermé.

Le secteur Dirac reste :

\[
\boxed{
R_{HH}^{D}\approx0
\qquad
\texttt{CONDITIONAL/STRUCTURAL}
}
\]

sur la branche générique :

\[
\det Q\neq0,
\qquad
\Delta_{\rm FF}\neq0.
\]

Le prédécesseur H8 a matérialisé un noyau \(\mathsf Y_{\rm encoded}\),
mais l'audit strict a identifié trois corrections covariantes nécessaires :

\[
\boxed{
\delta(D_k g_i)
=
D_k(\delta g_i)
-
\delta\Gamma^r{}_{ki}g_r
}
\]

\[
\boxed{
\delta(D_kq_{ij})
=
D_k(\delta q_{ij})
-
\delta\Gamma^r{}_{ki}q_{rj}
-
\delta\Gamma^r{}_{kj}q_{ir}
}
\]

et, puisque

\[
X_A\sim(X_{ij},X_s,X_i),
\]

\[
\boxed{
\delta(D_kX_{ij})
=
D_k(\delta X_{ij})
-
\delta\Gamma^r{}_{ki}X_{rj}
-
\delta\Gamma^r{}_{kj}X_{ir}
}
\]

\[
\boxed{
\delta(D_kX_i)
=
D_k(\delta X_i)
-
\delta\Gamma^r{}_{ki}X_r.
}
\]

Cette étape ne calcule **pas encore** le second jet métrique algébrique.

\[
\boxed{\texttt{DISPERSION\_READY=False}}
\]


In [ ]:

from __future__ import annotations

import sympy as sp
import json, sys
from pathlib import Path

print("GVH 0.3.2.7.3.7.3.3.1.6.2.1.1.1.1")
print("Python:", sys.version.split()[0])
print("SymPy:", sp.__version__)

UPSTREAM_CANONICAL_RHH = "STRONG_ZERO"
UPSTREAM_DIRAC_STATUS = "CONDITIONAL_STRUCTURAL_WEAK_ZERO_GENERIC_BRANCH"

RHH_PHYSICAL_CLASSIFIED = False
HYPERSURFACE_ALGEBRA_GLOBALLY_CLOSED = False
DISPERSION_READY = False

assert UPSTREAM_CANONICAL_RHH == "STRONG_ZERO"
assert not RHH_PHYSICAL_CLASSIFIED
assert not DISPERSION_READY

UPSTREAM_162111_EXECUTED_SHA256 = "54b89aab54e6e605f57319b565641582b6580b8c31a3a669b82a719f19a873ac"
print("UPSTREAM_162111_EXECUTED_SHA256 =",UPSTREAM_162111_EXECUTED_SHA256)


GVH 0.3.2.7.3.7.3.3.1.6.2.1.1.1.1
Python: 3.12.13
SymPy: 1.14.0
UPSTREAM_162111_EXECUTED_SHA256 = 54b89aab54e6e605f57319b565641582b6580b8c31a3a669b82a719f19a873ac



# 1 — Correction du test de Jacobi de `.1.6.2.1`

Dans `.1.6.2.1`, le coefficient de gradient du lapse avait été inséré
dans l'ansatz avant le contrôle Jacobi.

Ici on repart d'un ansatz réellement libre :

\[
\{\psi,H_0[N]\}
=
N\mathcal A_{\rm FF}
+
\alpha^iD_iN
+
\beta^{ij}D_iD_jN.
\]

Avec :

\[
\{H[N],H[M]\}_{\rm can}
=
D[\omega],
\qquad
\omega^i=ND^iM-MD^iN,
\]

et :

\[
\{\chi,H[N]\}=N\psi,
\]

l'identité de Jacobi doit déterminer indépendamment les structures
\(\alpha^i\) et \(\beta^{ij}\).


In [ ]:

N,M = sp.symbols("N M", real=True)

N1 = sp.Matrix(sp.symbols("N1:4", real=True))
M1 = sp.Matrix(sp.symbols("M1:4", real=True))

# six Hessiennes indépendantes : 11,22,33,12,13,23
N2s = sp.symbols("N11 N22 N33 N12 N13 N23", real=True)
M2s = sp.symbols("M11 M22 M33 M12 M13 M23", real=True)

alpha = sp.Matrix(sp.symbols("alpha1:4", real=True))
beta = sp.Matrix([
    [sp.Symbol("beta11"), sp.Symbol("beta12"), sp.Symbol("beta13")],
    [sp.Symbol("beta12"), sp.Symbol("beta22"), sp.Symbol("beta23")],
    [sp.Symbol("beta13"), sp.Symbol("beta23"), sp.Symbol("beta33")],
])

dchi = sp.Matrix(sp.symbols("dchi1:4", real=True))

N2 = sp.Matrix([
    [N2s[0],N2s[3],N2s[4]],
    [N2s[3],N2s[1],N2s[5]],
    [N2s[4],N2s[5],N2s[2]],
])

M2 = sp.Matrix([
    [M2s[0],M2s[3],M2s[4]],
    [M2s[3],M2s[1],M2s[5]],
    [M2s[4],M2s[5],M2s[2]],
])

Afree = sp.Symbol("A_FF_free", real=True)

psi_HN_ansatz = (
    N*Afree
    + alpha.dot(N1)
    + sum(beta[i,j]*N2[i,j] for i in range(3) for j in range(3))
)

psi_HM_ansatz = (
    M*Afree
    + alpha.dot(M1)
    + sum(beta[i,j]*M2[i,j] for i in range(3) for j in range(3))
)

omega_dot_dchi = sum(
    (N*M1[i]-M*N1[i])*dchi[i]
    for i in range(3)
)

jacobi = sp.expand(
    omega_dot_dchi
    + M*psi_HN_ansatz
    - N*psi_HM_ansatz
)

# Coefficients des Hessiennes arbitraires : ils imposent beta=0.
beta_constraints = []
for z in list(N2s)+list(M2s):
    beta_constraints.append(
        sp.expand(sp.diff(jacobi,z))
    )

# Coefficients des gradients : ils imposent alpha=dchi.
alpha_constraints = []
for i in range(3):
    alpha_constraints.append(
        sp.expand(
            jacobi.coeff(N*M1[i])
        )
    )

# Solution symbolique directe attendue.
jacobi_solution = {
    alpha[i]:dchi[i]
    for i in range(3)
}
for i in range(3):
    for j in range(i,3):
        jacobi_solution[beta[i,j]] = 0

jacobi_reduced = sp.expand(
    jacobi.subs(jacobi_solution)
)

assert jacobi_reduced == 0

JACOBI_INDEPENDENT_ANSATZ_USED = True
JACOBI_HESSIAN_COEFFICIENT_FORCED_ZERO = True
JACOBI_GRADIENT_COEFFICIENT_FORCED_DCHI = True

print("JACOBI_INDEPENDENT_ANSATZ_USED =",JACOBI_INDEPENDENT_ANSATZ_USED)
print("JACOBI_HESSIAN_COEFFICIENT_FORCED_ZERO =",JACOBI_HESSIAN_COEFFICIENT_FORCED_ZERO)
print("JACOBI_GRADIENT_COEFFICIENT_FORCED_DCHI =",JACOBI_GRADIENT_COEFFICIENT_FORCED_DCHI)


JACOBI_INDEPENDENT_ANSATZ_USED = True
JACOBI_HESSIAN_COEFFICIENT_FORCED_ZERO = True
JACOBI_GRADIENT_COEFFICIENT_FORCED_DCHI = True



Le résultat de ce contrôle est désormais réellement indépendant :

\[
\boxed{\beta^{ij}=0}
\]

et :

\[
\boxed{\alpha^i=D^i\chi}.
\]

Donc :

\[
\boxed{
\{\psi,H_0[N]\}
=
N\mathcal A_{\rm FF}
+
D^i\chi\,D_iN
}
\]

n'est plus seulement un ansatz compatible avec Jacobi.

Il reste cependant à calculer explicitement le coefficient local :

\[
\boxed{
\mathcal A_{\rm FF}
=
\{\psi,H_0[1]\}.
}
\]



# 2 — Reconstruction du même \(Q,J_0,U_0\)

On conserve exactement les conventions de `.1.6.2` et `.1.6.2.1`.

\[
V^A=
(K_{11},K_{22},K_{33},K_{12},K_{13},K_{23},S,W_1,W_2,W_3).
\]

Au point en coordonnées normales :

\[
h_{ij}=\delta_{ij},
\qquad
\Gamma^k{}_{ij}=0,
\]

mais les variations de connexion ne sont pas supprimées.


In [ ]:

c1,c2,c3,c4,s = sp.symbols("c1 c2 c3 c4 s", real=True)
v = sp.Matrix(sp.symbols("v1:4", real=True))
a = sp.Matrix(sp.symbols("a1:4", real=True))
g = sp.Matrix(sp.symbols("g1:4", real=True))

qsyms = sp.symbols(
    "q11 q12 q13 q21 q22 q23 q31 q32 q33",
    real=True
)
q = sp.Matrix(3,3,qsyms)

K11,K22,K33,K12,K13,K23,S,W1,W2,W3 = sp.symbols(
    "K11 K22 K33 K12 K13 K23 S W1 W2 W3",
    real=True
)

vel = sp.Matrix([
    K11,K22,K33,K12,K13,K23,S,W1,W2,W3
])

K = sp.Matrix([
    [K11,K12,K13],
    [K12,K22,K23],
    [K13,K23,K33],
])

W = sp.Matrix([W1,W2,W3])

A = -S-v.dot(a)
B = s*a+W-K*v
C = -g-K*v
D = q+s*K

I1 = (
    A**2-B.dot(B)-C.dot(C)
    +sum(D[i,j]**2 for i in range(3) for j in range(3))
)
theta = -A+sp.trace(D)
I3 = (
    A**2-2*B.dot(C)
    +sum(D[i,j]*D[j,i] for i in range(3) for j in range(3))
)
alpha0 = s*A+v.dot(C)
beta0 = s*B+D.T*v
acc2 = -alpha0**2+beta0.dot(beta0)

Lu = -c1*I1-c2*theta**2-c3*I3+c4*acc2
LEH = (
    sum(K[i,j]**2 for i in range(3) for j in range(3))
    -sp.trace(K)**2
)

Ltot = sp.expand(Lu+LEH)

zero_vel = {x:0 for x in vel}
zero_a = {x:0 for x in a}

Q = sp.hessian(Ltot,list(vel))
J0 = sp.Matrix([
    sp.diff(Ltot,x).subs(zero_vel)
    for x in vel
]).subs(zero_a)
U0 = sp.expand(
    Ltot.subs(zero_vel).subs(zero_a)
)

assert Q == Q.T
assert Q.shape == (10,10)

QJU_SAME_FULL_FIELD_RECONSTRUCTED = True
print("QJU_SAME_FULL_FIELD_RECONSTRUCTED =",QJU_SAME_FULL_FIELD_RECONSTRUCTED)


QJU_SAME_FULL_FIELD_RECONSTRUCTED = True



# 3 — Variables de Legendre \(X\) et duale \(Y\)

On n'expanse pas \(Q^{-1}\).

\[
\boxed{
QX=P-J_0
}
\]

et :

\[
\boxed{
QY=r
}
\]

avec :

\[
r=
(-2v_1^2,-2v_2^2,-2v_3^2,
-4v_1v_2,-4v_1v_3,-4v_2v_3,
-2s,2v_1,2v_2,2v_3)^T.
\]

Alors :

\[
\boxed{
\psi_{\rm FF}=-r^TX
}
\]

et :

\[
\boxed{
\Delta_{\rm FF}=-r^TY.
}
\]


In [ ]:

P = sp.Matrix(sp.symbols("P0:10", real=True))
X = sp.Matrix(sp.symbols("X0:10", real=True))
Y = sp.Matrix(sp.symbols("Y0:10", real=True))

r = sp.Matrix([
    -2*v[0]**2,
    -2*v[1]**2,
    -2*v[2]**2,
    -4*v[0]*v[1],
    -4*v[0]*v[2],
    -4*v[1]*v[2],
    -2*s,
    2*v[0],
    2*v[1],
    2*v[2],
])

psi_FF = sp.expand(-(r.T*X)[0])
Delta_FF = sp.expand(-(r.T*Y)[0])

LEGENDRE_RELATION = Q*X-(P-J0)
DUAL_RELATION = Q*Y-r

print("psi_FF length =",len(str(psi_FF)))
print("Delta_FF length =",len(str(Delta_FF)))


psi_FF length = 114
Delta_FF length = 114



# 4 — Jet spatial formel

Le coefficient \(\mathcal A_{\rm FF}\) contient les dérivées spatiales
des coefficients des variations fonctionnelles.

On introduit donc :

\[
D_iP_A,\quad
D_iX_A,\quad
D_iY_A,\quad
D_ig_j,\quad
D_iq_{jk}.
\]

Les relations dérivées de Legendre sont conservées sous forme de gates :

\[
D_i(QX-P+J_0)=0,
\qquad
D_i(QY-r)=0.
\]


In [ ]:

Dg = [[sp.Symbol(f"D{i}g{j}") for j in range(3)] for i in range(3)]
Dq = [[[sp.Symbol(f"D{i}q{j}{k}") for k in range(3)] for j in range(3)] for i in range(3)]
DP = [[sp.Symbol(f"D{i}P{A}") for A in range(10)] for i in range(3)]
DX = [[sp.Symbol(f"D{i}X{A}") for A in range(10)] for i in range(3)]
DY = [[sp.Symbol(f"D{i}Y{A}") for A in range(10)] for i in range(3)]

def Dop(expr,i):
    out = sp.diff(expr,s)*g[i]
    for j in range(3):
        out += sp.diff(expr,v[j])*q[i,j]
    for j in range(3):
        out += sp.diff(expr,g[j])*Dg[i][j]
    for j in range(3):
        for k in range(3):
            out += sp.diff(expr,q[j,k])*Dq[i][j][k]
    for A in range(10):
        out += sp.diff(expr,P[A])*DP[i][A]
        out += sp.diff(expr,X[A])*DX[i][A]
        out += sp.diff(expr,Y[A])*DY[i][A]
    return sp.expand(out)

SPATIAL_JET_OPERATOR_READY = True
print("SPATIAL_JET_OPERATOR_READY =",SPATIAL_JET_OPERATOR_READY)


SPATIAL_JET_OPERATOR_READY = True



# 5 — Dérivées compactes de \(C_N^{\rm loc}\) et de \(\psi_{\rm FF}\)

Pour toute variable locale \(z\) à \(P\) fixé :

\[
C_{,z}
=
J_{0,z}^TX
+
\frac12X^TQ_{,z}X
+
U_{0,z}.
\]

Pour \(\psi=-r^TX\) :

\[
\boxed{
\psi_{,z}
=
-r_{,z}^TX
+
Y^TJ_{0,z}
+
Y^TQ_{,z}X.
}
\]

Cette seconde formule vient de l'implicite :

\[
Q\,\delta X
=
\delta P-\delta J_0-\delta Q\,X.
\]

Aucun \(Q^{-1}\) symbolique géant n'est nécessaire.


In [ ]:

def Cpartial(z):
    return sp.expand(
        (J0.diff(z).T*X)[0]
        +sp.Rational(1,2)*(X.T*Q.diff(z)*X)[0]
        +sp.diff(U0,z)
    )

def Psipartial(z):
    return sp.expand(
        -(r.diff(z).T*X)[0]
        +(Y.T*J0.diff(z))[0]
        +(Y.T*Q.diff(z)*X)[0]
    )

Cg = sp.Matrix([Cpartial(g[i]) for i in range(3)])
Aq = sp.Matrix(3,3,[Cpartial(z) for z in list(q)])

Psi_g = sp.Matrix([Psipartial(g[i]) for i in range(3)])
Psi_q = sp.Matrix(3,3,[Psipartial(z) for z in list(q)])

Cs = Cpartial(s)
Cv = sp.Matrix([Cpartial(v[j]) for j in range(3)])

Psi_s = Psipartial(s)
Psi_v = sp.Matrix([Psipartial(v[j]) for j in range(3)])

Es = sp.expand(Cs-sum(Dop(Cg[i],i) for i in range(3)))
Ev = sp.Matrix([
    sp.expand(Cv[j]-sum(Dop(Aq[i,j],i) for i in range(3)))
    for j in range(3)
])

Epsi_s = sp.expand(
    Psi_s-sum(Dop(Psi_g[i],i) for i in range(3))
)

Epsi_v = sp.Matrix([
    sp.expand(
        Psi_v[j]-sum(Dop(Psi_q[i,j],i) for i in range(3))
    )
    for j in range(3)
])

SCALAR_VECTOR_FUNCTIONAL_JETS_EXPLICIT = True
print("SCALAR_VECTOR_FUNCTIONAL_JETS_EXPLICIT =",SCALAR_VECTOR_FUNCTIONAL_JETS_EXPLICIT)


SCALAR_VECTOR_FUNCTIONAL_JETS_EXPLICIT = True



# 6 — Jet métrique algébrique strict

Le terme local \(\mathcal A_{\rm FF}\) nécessite aussi les variations
métriques algébriques de \(C_N^{\rm loc}\) et de \(\psi_{\rm FF}\).

Cette cellule reconstruit les réponses au premier ordre autour de :

\[
h_{ij}=\delta_{ij},
\]

avec :

\[
h^{ij}=\delta^{ij}-\epsilon\,\delta h_{ij}.
\]

Le calcul peut être plus lourd que les cellules précédentes ; aucune
approximation n'est utilisée.


In [ ]:

e11,e22,e33,e12,e13,e23,eps = sp.symbols(
    "e11 e22 e33 e12 e13 e23 eps",
    real=True
)

dH = sp.Matrix([
    [e11,e12,e13],
    [e12,e22,e23],
    [e13,e23,e33],
])

hinv = sp.eye(3)-eps*dH
trdh = sp.trace(dH)

vup = hinv*v
Kmix = K*hinv

Ae = -S-(vup.T*a)[0]
Be = s*a+W-Kmix*v
Ce = -g-Kmix*v
De = q+s*K

def cdot(x,y):
    return (x.T*hinv*y)[0]

I1e = (
    Ae**2
    -cdot(Be,Be)
    -cdot(Ce,Ce)
    +sp.trace(hinv*De*hinv*De.T)
)

thetae = -Ae+sp.trace(hinv*De)

I3e = (
    Ae**2
    -2*cdot(Be,Ce)
    +sp.trace(hinv*De*hinv*De)
)

alphae = s*Ae+(vup.T*Ce)[0]
betae = s*Be+De.T*vup
acc2e = -alphae**2+cdot(betae,betae)

Lue = -c1*I1e-c2*thetae**2-c3*I3e+c4*acc2e
LEHe = (
    sp.trace(hinv*K*hinv*K.T)
    -sp.trace(hinv*K)**2
)

Ltote = Lue+LEHe

dL = sp.diff(Ltote,eps).subs(eps,0)

dQ = sp.hessian(dL,list(vel)).subs(zero_a)
dJ = sp.Matrix([
    sp.diff(dL,z).subs(zero_vel).subs(zero_a)
    for z in vel
])
dU = sp.diff(dL.subs(zero_vel).subs(zero_a),eps) if dL.has(eps) else 0

# dL ne dépend plus de eps ; U-variation directe :
dU = dL.subs(zero_vel).subs(zero_a)

deltaP = sp.Matrix([
    -sp.Rational(1,2)*trdh*P[A]
    for A in range(10)
])

CNloc = (
    -sp.Rational(1,2)*((P-J0).T*X)[0]
    +U0
)

dCN_density = (
    (dJ.T*X)[0]
    +sp.Rational(1,2)*(X.T*dQ*X)[0]
    +dU
    -(X.T*deltaP)[0]
    +sp.Rational(1,2)*trdh*CNloc
)

# r(h) au premier ordre
re = sp.Matrix([
    -2*vup[0]**2,
    -2*vup[1]**2,
    -2*vup[2]**2,
    -4*vup[0]*vup[1],
    -4*vup[0]*vup[2],
    -4*vup[1]*vup[2],
    -2*s,
    2*vup[0],
    2*vup[1],
    2*vup[2],
])

dr = sp.Matrix([
    sp.diff(re[A],eps).subs(eps,0)
    for A in range(10)
])

dpsi_density = (
    sp.Rational(1,2)*trdh*psi_FF
    -(dr.T*X)[0]
    +(Y.T*dJ)[0]
    +(Y.T*dQ*X)[0]
    -(Y.T*deltaP)[0]
)

metric_vars = [e11,e22,e33,e12,e13,e23]

cCN = [sp.diff(dCN_density,e) for e in metric_vars]
cPSI = [sp.diff(dpsi_density,e) for e in metric_vars]

T_C = sp.Matrix([
    [cCN[0],cCN[3]/2,cCN[4]/2],
    [cCN[3]/2,cCN[1],cCN[5]/2],
    [cCN[4]/2,cCN[5]/2,cCN[2]],
])

T_PSI = sp.Matrix([
    [cPSI[0],cPSI[3]/2,cPSI[4]/2],
    [cPSI[3]/2,cPSI[1],cPSI[5]/2],
    [cPSI[4]/2,cPSI[5]/2,cPSI[2]],
])

assert T_C == T_C.T
assert T_PSI == T_PSI.T

METRIC_ALGEBRAIC_JETS_EXPLICIT = True
print("METRIC_ALGEBRAIC_JETS_EXPLICIT =",METRIC_ALGEBRAIC_JETS_EXPLICIT)
print("T_C lengths =",[len(str(T_C[i,j])) for i in range(3) for j in range(i,3)])
print("T_PSI lengths =",[len(str(T_PSI[i,j])) for i in range(3) for j in range(i,3)])


METRIC_ALGEBRAIC_JETS_EXPLICIT = True
T_C lengths = [4972, 5279, 5279, 4972, 5280, 4972]
T_PSI lengths = [2639, 4758, 4758, 2639, 4757, 2639]



# 7 — Variation de connexion : termes locaux pour \(N=1\)

Pour une quantité locale dont le \(q_{ij}=D_iv_j\) jet vaut
\(\mathcal B^{ij}\), la variation de connexion fournit :

\[
\mathcal K_\Gamma^{mn}[f]
=
\kappa_i{}^{mn}D_if
+
f\,\kappa_0{}^{mn},
\]

où \(\kappa_i{}^{mn}\) est le coefficient déjà dérivé dans `.1.6.1`.

Le coefficient \(\kappa_0{}^{mn}\) est ici calculé explicitement par
la dérivée spatiale des coefficients.

Il est nécessaire pour :

\[
\mathcal A_{\rm FF}=\{\psi,H_0[1]\}.
\]


In [ ]:

def connection_blocks(Bjet):
    kgrad = [
        [
            [
                sp.expand(
                    sp.Rational(1,4)*(
                        Bjet[i,m]*v[n]
                        +Bjet[i,n]*v[m]
                        +Bjet[m,i]*v[n]
                        +Bjet[n,i]*v[m]
                        -(Bjet[m,n]+Bjet[n,m])*v[i]
                    )
                )
                for n in range(3)
            ]
            for m in range(3)
        ]
        for i in range(3)
    ]

    klocal = sp.Matrix(
        3,3,
        lambda m,n: sp.expand(
            sum(
                sp.Rational(1,4)*Dop(
                    Bjet[i,m]*v[n]+Bjet[i,n]*v[m],
                    i
                )
                for i in range(3)
            )
            +
            sum(
                sp.Rational(1,4)*Dop(
                    Bjet[m,j]*v[n]+Bjet[n,j]*v[m],
                    j
                )
                for j in range(3)
            )
            -
            sum(
                sp.Rational(1,4)*Dop(
                    (Bjet[m,n]+Bjet[n,m])*v[l],
                    l
                )
                for l in range(3)
            )
        )
    )

    return kgrad,klocal

Kgrad_C,K0_C = connection_blocks(Aq)
Kgrad_PSI,K0_PSI = connection_blocks(Psi_q)

CONNECTION_LOCAL_BLOCKS_EXPLICIT = True
print("CONNECTION_LOCAL_BLOCKS_EXPLICIT =",CONNECTION_LOCAL_BLOCKS_EXPLICIT)


CONNECTION_LOCAL_BLOCKS_EXPLICIT = True



# 8 — Matérialisation de \(\mathcal A_{\rm FF}\)

On évalue directement :

\[
\boxed{
\mathcal A_{\rm FF}
=
\{\Psi[f],H_0[1]\}
}
\]

puis on intègre par parties les termes \(D_if\).

Le tenseur d'Einstein spatial ultralocal est conservé explicitement :

\[
G^{ij}_{(3)}.
\]

Il n'est pas mis à zéro par le choix de coordonnées normales.


In [ ]:

G11,G22,G33,G12,G13,G23 = sp.symbols(
    "G11 G22 G33 G12 G13 G23",
    real=True
)

G3 = sp.Matrix([
    [G11,G12,G13],
    [G12,G22,G23],
    [G13,G23,G33],
])

metric_pairs = [
    (0,0),(1,1),(2,2),
    (0,1),(0,2),(1,2),
]
metric_weights = [1,1,1,2,2,2]

K00 = sp.Integer(0)
Kf1 = [sp.Integer(0) for _ in range(3)]

# Secteur métrique
for A,(m,n) in enumerate(metric_pairs):
    w = metric_weights[A]

    psi_q0 = w*(T_PSI[m,n]+K0_PSI[m,n])
    psi_q1 = [w*Kgrad_PSI[i][m][n] for i in range(3)]
    psi_p0 = -2*Y[A]

    H_q0 = w*(T_C[m,n]+K0_C[m,n]-G3[m,n])
    H_p0 = -2*X[A]

    K00 += psi_q0*H_p0-psi_p0*H_q0

    for i in range(3):
        Kf1[i] += psi_q1[i]*H_p0

# Secteur scalaire
psi_q0 = Epsi_s
psi_q1 = [-Psi_g[i] for i in range(3)]
psi_p0 = -Y[6]

H_q0 = Es
H_p0 = -X[6]

K00 += psi_q0*H_p0-psi_p0*H_q0
for i in range(3):
    Kf1[i] += psi_q1[i]*H_p0

# Secteur vectoriel
for j in range(3):
    psi_q0 = Epsi_v[j]
    psi_q1 = [-Psi_q[i,j] for i in range(3)]
    psi_p0 = -Y[7+j]

    H_q0 = Ev[j]
    H_p0 = -X[7+j]

    K00 += psi_q0*H_p0-psi_p0*H_q0

    for i in range(3):
        Kf1[i] += psi_q1[i]*H_p0

A_FF = sp.expand(
    K00-sum(Dop(Kf1[i],i) for i in range(3))
)

A_FF_MATERIALIZED_FROM_QJU = True

print("A_FF_MATERIALIZED_FROM_QJU =",A_FF_MATERIALIZED_FROM_QJU)
print("A_FF expression length =",len(str(A_FF)))


A_FF_MATERIALIZED_FROM_QJU = True
A_FF expression length = 55337



# 9 — \(\rho_{\rm FF}\) explicite

Avec :

\[
H=H_0-\lambda_{\rm mult}\chi,
\]

et :

\[
\Delta_{\rm FF}=\{\chi,\psi\},
\]

la quatrième contrainte est maintenant définie par un objet réellement
calculé :

\[
\boxed{
\rho_{\rm FF}
=
\mathcal A_{\rm FF}
+
\lambda_{\rm mult}\Delta_{\rm FF}.
}
\]

Contrairement à `.1.6.2.1`, \(\mathcal A_{\rm FF}\) n'est plus un symbole libre.


In [ ]:

lambda_mult = sp.Symbol("lambda_mult", real=True)

rho_FF = sp.expand(
    A_FF+lambda_mult*Delta_FF
)

RHO_FF_MATERIALIZED_FROM_QJU = (
    A_FF_MATERIALIZED_FROM_QJU
)

assert not rho_FF.has(sp.Symbol("A_FF_free"))

print("RHO_FF_MATERIALIZED_FROM_QJU =",RHO_FF_MATERIALIZED_FROM_QJU)
print("rho_FF expression length =",len(str(rho_FF)))


RHO_FF_MATERIALIZED_FROM_QJU = True
rho_FF expression length = 55574



# 10 — Distribution \(\mathsf X(x,y)=\{\chi(x),\rho(y)\}\)

Comme \(\chi\) ne dépend d'aucun moment canonique, son flot dans le
calcul de \(\{\chi,\rho\}\) agit seulement dans les directions
impulsionnelles.

Dans les variables compactes :

\[
\delta_\chi P_A=r_A\,\delta,
\]

\[
\delta_\chi X_A=Y_A\,\delta,
\]

puis :

\[
\delta_\chi(D_iP_A)
=
(D_ir_A)\delta+r_A D_i\delta,
\]

\[
\delta_\chi(D_iX_A)
=
(D_iY_A)\delta+Y_A D_i\delta.
\]

Ainsi le noyau doit avoir l'ordre :

\[
\boxed{
\mathsf X(x,y)
=
X_0(y)\delta(x-y)
+
X_1^i(y)D_i\delta(x-y).
}
\]

Les coefficients sont calculés ci-dessous directement depuis
\(\rho_{\rm FF}\).


In [ ]:

Dr = [
    sp.Matrix([Dop(r[A],i) for A in range(10)])
    for i in range(3)
]

Xker0 = sp.Integer(0)
Xker1 = [sp.Integer(0) for _ in range(3)]

for A in range(10):
    Xker0 += sp.diff(rho_FF,P[A])*r[A]
    Xker0 += sp.diff(rho_FF,X[A])*Y[A]

    for i in range(3):
        Xker0 += sp.diff(rho_FF,DP[i][A])*Dr[i][A]
        Xker0 += sp.diff(rho_FF,DX[i][A])*DY[i][A]

        Xker1[i] += sp.diff(rho_FF,DP[i][A])*r[A]
        Xker1[i] += sp.diff(rho_FF,DX[i][A])*Y[A]

Xker0 = sp.expand(Xker0)
Xker1 = [sp.expand(z) for z in Xker1]

X_DISTRIBUTIONAL_KERNEL_MATERIALIZED = True

print("X_DISTRIBUTIONAL_KERNEL_MATERIALIZED =",X_DISTRIBUTIONAL_KERNEL_MATERIALIZED)
print("X kernel delta length =",len(str(Xker0)))
print("X kernel Ddelta lengths =",[len(str(z)) for z in Xker1])


X_DISTRIBUTIONAL_KERNEL_MATERIALIZED = True
X kernel delta length = 33695
X kernel Ddelta lengths = [1202, 1201, 1201]



# 11 — Convention d'adjoint distributionnel

Pour un opérateur :

\[
L
=
a(x)+b^i(x)D_i,
\]

l'adjoint formel sous :

\[
\langle f,g\rangle
=
\int_\Sigma d^3x\,\sqrt h\,f\,g
\]

et avec conditions de bord annulant le flux est :

\[
\boxed{
L^\dagger
=
a-D_i b^i-b^iD_i.
}
\]

Les conditions autorisées dans ce notebook sont :

- \(\Sigma\) compacte sans bord ; ou
- champs/tests à décroissance suffisante ; ou
- conditions de bord telles que les termes de flux s'annulent.

Aucune autre condition n'est supposée.


In [ ]:

BOUNDARY_DOMAIN = (
    "compact_without_boundary OR sufficient_decay "
    "OR boundary_conditions_killing_flux"
)

X_ADJOINT_FORMALIZED = X_DISTRIBUTIONAL_KERNEL_MATERIALIZED

print("BOUNDARY_DOMAIN =",BOUNDARY_DOMAIN)
print("X_ADJOINT_FORMALIZED =",X_ADJOINT_FORMALIZED)


BOUNDARY_DOMAIN = compact_without_boundary OR sufficient_decay OR boundary_conditions_killing_flux
X_ADJOINT_FORMALIZED = True



# 12 — Audit strict de \(\mathsf Y(x,y)=\{\psi(x),\rho(y)\}\)

Ici la situation est plus exigeante.

Le flot de \(\psi\) modifie :

\[
h_{ij},\quad s,\quad v_i,
\]

et leurs moments. Par conséquent il modifie également :

\[
g_i=D_is,\qquad
q_{ij}=D_iv_j,
\qquad
G^{ij}_{(3)}.
\]

La variation de \(q_{ij}\) contient :

\[
\delta q_{ij}
=
D_i(\delta v_j)
-
\delta\Gamma^k{}_{ij}v_k,
\]

et la variation de \(G^{ij}_{(3)}\) contient des dérivées secondes de
\(\delta h_{mn}\).

Par conséquent le vrai noyau \(\mathsf Y\) peut contenir :

\[
\delta,
\quad
D_i\delta,
\quad
D_iD_j\delta,
\quad
D_iD_jD_k\delta
\]

selon l'ordre final du jet de \(\rho_{\rm FF}\).

Cette cellule interdit donc de remplacer \(\mathsf Y\) par une matrice
finie-dimensionnelle arbitraire.


In [ ]:

# Détection objective des dépendances qui obligent à inclure
# la variation de courbure et les jets de connexion.

rho_dependencies = {
    "contains_spatial_Einstein_tensor":
        any(rho_FF.has(z) for z in [G11,G22,G33,G12,G13,G23]),

    "contains_Dg":
        any(rho_FF.has(Dg[i][j]) for i in range(3) for j in range(3)),

    "contains_Dq":
        any(
            rho_FF.has(Dq[i][j][k])
            for i in range(3)
            for j in range(3)
            for k in range(3)
        ),

    "contains_DP":
        any(rho_FF.has(DP[i][A]) for i in range(3) for A in range(10)),

    "contains_DX":
        any(rho_FF.has(DX[i][A]) for i in range(3) for A in range(10)),

    "contains_DY":
        any(rho_FF.has(DY[i][A]) for i in range(3) for A in range(10)),
}

for k,vv in rho_dependencies.items():
    print(k,":",vv)

Y_REQUIRES_CURVATURE_FRECHET_JET = (
    rho_dependencies["contains_spatial_Einstein_tensor"]
)

Y_REQUIRES_CONNECTION_FRECHET_JET = (
    rho_dependencies["contains_Dq"]
)

print("Y_REQUIRES_CURVATURE_FRECHET_JET =",Y_REQUIRES_CURVATURE_FRECHET_JET)
print("Y_REQUIRES_CONNECTION_FRECHET_JET =",Y_REQUIRES_CONNECTION_FRECHET_JET)


contains_spatial_Einstein_tensor : True
contains_Dg : True
contains_Dq : True
contains_DP : False
contains_DX : True
contains_DY : False
Y_REQUIRES_CURVATURE_FRECHET_JET = True
Y_REQUIRES_CONNECTION_FRECHET_JET = True



# 13 — Flot canonique de \(\psi\) sur les variables de configuration

Pour une fonction test \(f\), écrivons :

\[
\Psi[f]=\int_\Sigma d^3x\,\sqrt h\,f\,\psi_{\rm FF}.
\]

Les dérivées impulsionnelles de \(\psi\) donnent le flot de configuration :

\[
\delta_\psi h_{ij}=f\,H^{(\psi)}_{ij},
\qquad
\delta_\psi s=f\,S_\psi,
\qquad
\delta_\psi v_i=f\,V^{(\psi)}_i.
\]

Dans les conventions canoniques indépendantes utilisées dans la chaîne :

\[
\boxed{
H^{(\psi)}=
-2
\begin{pmatrix}
Y_0&Y_3&Y_4\\
Y_3&Y_1&Y_5\\
Y_4&Y_5&Y_2
\end{pmatrix}
}
\]

et :

\[
\boxed{
S_\psi=-Y_6,
\qquad
V_i^{(\psi)}=-Y_{7+i}.
}
\]


In [ ]:

Hpsi = sp.Matrix([
    [-2*Y[0], -2*Y[3], -2*Y[4]],
    [-2*Y[3], -2*Y[1], -2*Y[5]],
    [-2*Y[4], -2*Y[5], -2*Y[2]],
])

Spsi = -Y[6]
Vpsi = sp.Matrix([-Y[7],-Y[8],-Y[9]])

D2Y = [
    [
        [sp.Symbol(f"D2Y{i}{j}_{A}") for A in range(10)]
        for j in range(3)
    ]
    for i in range(3)
]

def H_from_components(vals):
    return sp.Matrix([
        [vals[0],vals[3],vals[4]],
        [vals[3],vals[1],vals[5]],
        [vals[4],vals[5],vals[2]],
    ])

DHpsi = [
    H_from_components([-2*DY[i][A] for A in range(6)])
    for i in range(3)
]

D2Hpsi = [
    [
        H_from_components([-2*D2Y[i][j][A] for A in range(6)])
        for j in range(3)
    ]
    for i in range(3)
]

CONFIGURATION_PSI_FLOW_EXPLICIT = True

print("CONFIGURATION_PSI_FLOW_EXPLICIT =",CONFIGURATION_PSI_FLOW_EXPLICIT)


CONFIGURATION_PSI_FLOW_EXPLICIT = True



# 14 — Variation de connexion sous le flot de \(\psi\)

Au point en coordonnées normales :

\[
\delta_\psi\Gamma^k{}_{ij}
=
\frac12
\left(
D_i\delta h_j{}^k
+
D_j\delta h_i{}^k
-
D^k\delta h_{ij}
\right).
\]

Avec :

\[
\delta h_{ij}=fH^{(\psi)}_{ij},
\]

on sépare exactement :

\[
\boxed{
\delta_\psi\Gamma^k{}_{ij}
=
f\,\Gamma^{k}{}_{ij|0}
+
(D_\ell f)\,
\Gamma^{k\ell}{}_{ij|1}.
}
\]

Cela permet ensuite de calculer :

\[
\delta_\psi q_{ij}
=
\delta_\psi(D_iv_j)
=
D_i(\delta_\psi v_j)
-
\delta_\psi\Gamma^k{}_{ij}v_k.
\]


In [ ]:

Gamma0 = [
    [
        [
            sp.expand(
                sp.Rational(1,2)*(
                    DHpsi[i][j,k]
                    +DHpsi[j][i,k]
                    -DHpsi[k][i,j]
                )
            )
            for j in range(3)
        ]
        for i in range(3)
    ]
    for k in range(3)
]

Gamma1 = [
    [
        [
            [
                sp.expand(
                    sp.Rational(1,2)*(
                        (1 if i==l else 0)*Hpsi[j,k]
                        +(1 if j==l else 0)*Hpsi[i,k]
                        -(1 if k==l else 0)*Hpsi[i,j]
                    )
                )
                for j in range(3)
            ]
            for i in range(3)
        ]
        for k in range(3)
    ]
    for l in range(3)
]

gflow0 = [-DY[j][6] for j in range(3)]
gflow1 = [
    [
        (Spsi if j==l else sp.Integer(0))
        for j in range(3)
    ]
    for l in range(3)
]

qflow0 = [
    [
        sp.expand(
            -DY[i][7+j]
            -sum(Gamma0[k][i][j]*v[k] for k in range(3))
        )
        for j in range(3)
    ]
    for i in range(3)
]

qflow1 = [
    [
        [
            sp.expand(
                ((Vpsi[j]) if i==l else 0)
                -sum(Gamma1[l][k][i][j]*v[k] for k in range(3))
            )
            for j in range(3)
        ]
        for i in range(3)
    ]
    for l in range(3)
]

CONNECTION_FRECHET_Q_JET_EXPLICIT = True
print("CONNECTION_FRECHET_Q_JET_EXPLICIT =",CONNECTION_FRECHET_Q_JET_EXPLICIT)


CONNECTION_FRECHET_Q_JET_EXPLICIT = True



# 15 — Correction covariante de \(D_k g_i\) et \(D_kq_{ij}\)

On repart des blocs déjà établis :

\[
\delta_\psi\Gamma^r{}_{ki}
=
f\,\Gamma^r{}_{ki|0}
+
(D_\ell f)\Gamma^{r\ell}{}_{ki|1}.
\]

Le terme externe de connexion est ajouté **après**
la dérivation de \(\delta g_i\) ou \(\delta q_{ij}\).

Pour \(D_kg_i\) :

\[
\delta(D_kg_i)
=
f\,G^{(0)}_{ki}
+
(D_\ell f)G^{(1)\ell}_{ki}
+
(D_aD_bf)G^{(2)ab}_{ki}.
\]

Pour \(D_kq_{ij}\) :

\[
\delta(D_kq_{ij})
=
f\,Q^{(0)}_{kij}
+
(D_\ell f)Q^{(1)\ell}_{kij}
+
(D_aD_bf)Q^{(2)ab}_{kij}.
\]


In [ ]:

# ---------- Uncorrected derivative pieces, retained as audit witnesses ----------

Dgflow0_derivative_only = [
    [
        -D2Y[k][j][6]
        for j in range(3)
    ]
    for k in range(3)
]

Dgflow1_derivative_only = [
    [
        [
            sp.expand(
                (gflow0[j] if k==l else 0)
                +((-DY[k][6]) if j==l else 0)
            )
            for j in range(3)
        ]
        for k in range(3)
    ]
    for l in range(3)
]

Dgflow2 = [
    [
        [
            [
                (Spsi if (k==a and j==b) else sp.Integer(0))
                for j in range(3)
            ]
            for k in range(3)
        ]
        for b in range(3)
    ]
    for a in range(3)
]

# ---------- D_k Gamma blocks ----------

DGamma0 = [
    [
        [
            [
                sp.expand(
                    sp.Rational(1,2)*(
                        D2Hpsi[k][i][j,r]
                        +D2Hpsi[k][j][i,r]
                        -D2Hpsi[k][r][i,j]
                    )
                )
                for j in range(3)
            ]
            for i in range(3)
        ]
        for r in range(3)
    ]
    for k in range(3)
]

DGamma1 = [
    [
        [
            [
                [
                    sp.expand(
                        sp.Rational(1,2)*(
                            (1 if i==l else 0)*DHpsi[k][j,r]
                            +(1 if j==l else 0)*DHpsi[k][i,r]
                            -(1 if r==l else 0)*DHpsi[k][i,j]
                        )
                    )
                    for j in range(3)
                ]
                for i in range(3)
            ]
            for r in range(3)
        ]
        for k in range(3)
    ]
    for l in range(3)
]

# ---------- Covariant correction for D_k g_i ----------

Dgflow0_external = [
    [
        sp.expand(
            -sum(
                Gamma0[r][k][i]*g[r]
                for r in range(3)
            )
        )
        for i in range(3)
    ]
    for k in range(3)
]

Dgflow1_external = [
    [
        [
            sp.expand(
                -sum(
                    Gamma1[l][r][k][i]*g[r]
                    for r in range(3)
                )
            )
            for i in range(3)
        ]
        for k in range(3)
    ]
    for l in range(3)
]

Dgflow0 = [
    [
        sp.expand(
            Dgflow0_derivative_only[k][i]
            +Dgflow0_external[k][i]
        )
        for i in range(3)
    ]
    for k in range(3)
]

Dgflow1 = [
    [
        [
            sp.expand(
                Dgflow1_derivative_only[l][k][i]
                +Dgflow1_external[l][k][i]
            )
            for i in range(3)
        ]
        for k in range(3)
    ]
    for l in range(3)
]

# ---------- Derivative-only D_k q_ij pieces ----------

Dqflow0_derivative_only = [
    [
        [
            sp.expand(
                -D2Y[k][i][7+j]
                -sum(
                    DGamma0[k][r][i][j]*v[r]
                    +Gamma0[r][i][j]*q[k,r]
                    for r in range(3)
                )
            )
            for j in range(3)
        ]
        for i in range(3)
    ]
    for k in range(3)
]

Dqflow1_derivative_only = [
    [
        [
            [
                sp.expand(
                    (qflow0[i][j] if k==l else 0)
                    +((-DY[k][7+j]) if i==l else 0)
                    -sum(
                        DGamma1[l][k][r][i][j]*v[r]
                        +Gamma1[l][r][i][j]*q[k,r]
                        for r in range(3)
                    )
                )
                for j in range(3)
            ]
            for i in range(3)
        ]
        for k in range(3)
    ]
    for l in range(3)
]

Dqflow2 = [
    [
        [
            [
                [
                    (qflow1[b][i][j] if k==a else sp.Integer(0))
                    for j in range(3)
                ]
                for i in range(3)
            ]
            for k in range(3)
        ]
        for b in range(3)
    ]
    for a in range(3)
]

# ---------- External connection terms for D_k q_ij ----------

Dqflow0_external = [
    [
        [
            sp.expand(
                -sum(
                    Gamma0[r][k][i]*q[r,j]
                    +Gamma0[r][k][j]*q[i,r]
                    for r in range(3)
                )
            )
            for j in range(3)
        ]
        for i in range(3)
    ]
    for k in range(3)
]

Dqflow1_external = [
    [
        [
            [
                sp.expand(
                    -sum(
                        Gamma1[l][r][k][i]*q[r,j]
                        +Gamma1[l][r][k][j]*q[i,r]
                        for r in range(3)
                    )
                )
                for j in range(3)
            ]
            for i in range(3)
        ]
        for k in range(3)
    ]
    for l in range(3)
]

Dqflow0 = [
    [
        [
            sp.expand(
                Dqflow0_derivative_only[k][i][j]
                +Dqflow0_external[k][i][j]
            )
            for j in range(3)
        ]
        for i in range(3)
    ]
    for k in range(3)
]

Dqflow1 = [
    [
        [
            [
                sp.expand(
                    Dqflow1_derivative_only[l][k][i][j]
                    +Dqflow1_external[l][k][i][j]
                )
                for j in range(3)
            ]
            for i in range(3)
        ]
        for k in range(3)
    ]
    for l in range(3)
]

# Exact regression identities for every component.
assert all(
    sp.expand(
        Dgflow0[k][i]
        -Dgflow0_derivative_only[k][i]
        -Dgflow0_external[k][i]
    ) == 0
    for k in range(3)
    for i in range(3)
)

assert all(
    sp.expand(
        Dgflow1[l][k][i]
        -Dgflow1_derivative_only[l][k][i]
        -Dgflow1_external[l][k][i]
    ) == 0
    for l in range(3)
    for k in range(3)
    for i in range(3)
)

assert all(
    sp.expand(
        Dqflow0[k][i][j]
        -Dqflow0_derivative_only[k][i][j]
        -Dqflow0_external[k][i][j]
    ) == 0
    for k in range(3)
    for i in range(3)
    for j in range(3)
)

assert all(
    sp.expand(
        Dqflow1[l][k][i][j]
        -Dqflow1_derivative_only[l][k][i][j]
        -Dqflow1_external[l][k][i][j]
    ) == 0
    for l in range(3)
    for k in range(3)
    for i in range(3)
    for j in range(3)
)

COVARIANT_EXTERNAL_CONNECTION_DG_CORRECTED = True
COVARIANT_EXTERNAL_CONNECTION_DQ_CORRECTED = True

print(
    "COVARIANT_EXTERNAL_CONNECTION_DG_CORRECTED =",
    COVARIANT_EXTERNAL_CONNECTION_DG_CORRECTED
)
print(
    "COVARIANT_EXTERNAL_CONNECTION_DQ_CORRECTED =",
    COVARIANT_EXTERNAL_CONNECTION_DQ_CORRECTED
)


COVARIANT_EXTERNAL_CONNECTION_DG_CORRECTED = True
COVARIANT_EXTERNAL_CONNECTION_DQ_CORRECTED = True



# 16 — Variation de Fréchet du tenseur d'Einstein spatial

On utilise la formule covariante non-commutée :

\[
\delta R_{ij}
=
\frac12
\left(
D_kD_i h_j{}^k
+
D_kD_j h_i{}^k
-
D^2h_{ij}
-
D_jD_i h
\right),
\]

\[
\delta R
=
-h^{ij}R_{ij}
+
D_iD_jh^{ij}
-
D^2h,
\]

et :

\[
\delta G^{mn}
=
\delta R^{mn}
-\frac12\,\delta h^{mn}R
-\frac12 h^{mn}\delta R.
\]

Au point \(h_{ij}=\delta_{ij}\), le Ricci de fond est reconstruit depuis
\(G^{ij}_{(3)}\) :

\[
R=-2\,G^i{}_i,
\qquad
R_{ij}=G_{ij}-\delta_{ij}G^k{}_k.
\]

La variation est décomposée en coefficients de :

\[
f,\qquad D_if,\qquad D_iD_jf.
\]


In [ ]:

trG = sp.trace(G3)
Rsc = sp.expand(-2*trG)
Ric = sp.expand(G3-sp.eye(3)*trG)

# Second covariant derivative of f*H:
# returns coefficients at orders f, D_l f, D_lD_m f.
def DD_fH(aidx,bidx,cidx,didx):
    c0 = D2Hpsi[aidx][bidx][cidx,didx]
    c1 = [
        sp.expand(
            (DHpsi[bidx][cidx,didx] if l==aidx else 0)
            +(DHpsi[aidx][cidx,didx] if l==bidx else 0)
        )
        for l in range(3)
    ]
    c2 = [
        [
            (
                Hpsi[cidx,didx]
                if (l==aidx and m==bidx)
                else sp.Integer(0)
            )
            for m in range(3)
        ]
        for l in range(3)
    ]
    return c0,c1,c2

dRicci0 = sp.zeros(3)
dRicci1 = [sp.zeros(3) for _ in range(3)]
dRicci2 = [[sp.zeros(3) for _ in range(3)] for __ in range(3)]

for i in range(3):
    for j in range(3):
        c0 = sp.Integer(0)
        c1 = [sp.Integer(0) for _ in range(3)]
        c2 = [[sp.Integer(0) for _ in range(3)] for __ in range(3)]

        for k in range(3):
            terms = [
                (1, DD_fH(k,i,j,k)),
                (1, DD_fH(k,j,i,k)),
                (-1, DD_fH(k,k,i,j)),
                (-1, DD_fH(j,i,k,k)),
            ]
            for sign,(z0,z1,z2) in terms:
                c0 += sp.Rational(sign,2)*z0
                for l in range(3):
                    c1[l] += sp.Rational(sign,2)*z1[l]
                    for m in range(3):
                        c2[l][m] += sp.Rational(sign,2)*z2[l][m]

        dRicci0[i,j] = sp.expand(c0)
        for l in range(3):
            dRicci1[l][i,j] = sp.expand(c1[l])
            for m in range(3):
                dRicci2[l][m][i,j] = sp.expand(c2[l][m])

dR0 = sp.expand(
    -sum(Hpsi[i,j]*Ric[i,j] for i in range(3) for j in range(3))
    +sp.trace(dRicci0)
)
dR1 = [
    sp.expand(sp.trace(dRicci1[l]))
    for l in range(3)
]
dR2 = [
    [
        sp.expand(sp.trace(dRicci2[l][m]))
        for m in range(3)
    ]
    for l in range(3)
]

dG0 = sp.zeros(3)
dG1 = [sp.zeros(3) for _ in range(3)]
dG2 = [[sp.zeros(3) for _ in range(3)] for __ in range(3)]

for m in range(3):
    for n in range(3):
        dG0[m,n] = sp.expand(
            dRicci0[m,n]
            -sum(Hpsi[m,k]*Ric[k,n] for k in range(3))
            -sum(Hpsi[n,k]*Ric[m,k] for k in range(3))
            +sp.Rational(1,2)*Hpsi[m,n]*Rsc
            -sp.Rational(1,2)*(1 if m==n else 0)*dR0
        )

        for l in range(3):
            dG1[l][m,n] = sp.expand(
                dRicci1[l][m,n]
                -sp.Rational(1,2)*(1 if m==n else 0)*dR1[l]
            )
            for r2 in range(3):
                dG2[l][r2][m,n] = sp.expand(
                    dRicci2[l][r2][m,n]
                    -sp.Rational(1,2)*(1 if m==n else 0)*dR2[l][r2]
                )

CURVATURE_FRECHET_SECOND_VARIATION_CONTRACTED = True

print(
    "CURVATURE_FRECHET_SECOND_VARIATION_CONTRACTED =",
    CURVATURE_FRECHET_SECOND_VARIATION_CONTRACTED
)


CURVATURE_FRECHET_SECOND_VARIATION_CONTRACTED = True



# 17 — Flot impulsionnel et flot de \(P_A\)

Le flot impulsionnel de \(\psi\) est obtenu depuis les mêmes dérivées
fonctionnelles déjà matérialisées.

Pour les six paires métriques :

\[
\delta_\psi\Pi_A
=
-f\,\Psi^{(0)}_{h_A}
-
(D_if)\,\Psi^{(1)i}_{h_A}.
\]

Pour \(p_s,p_v^i\), on utilise les jets d'Euler déjà calculés.

Le passage aux moments normalisés \(P_A\) conserve aussi :

\[
\delta(1/\sqrt h)
=
-\frac12\,h^{ij}\delta h_{ij}.
\]


In [ ]:

pflow0 = [sp.Integer(0) for _ in range(10)]
pflow1 = [[sp.Integer(0) for _ in range(10)] for __ in range(3)]

# metric canonical momenta Pi_A
for Aidx,(m,n) in enumerate(metric_pairs):
    w = metric_weights[Aidx]
    pflow0[Aidx] = sp.expand(
        -w*(T_PSI[m,n]+K0_PSI[m,n])
    )
    for i in range(3):
        pflow1[i][Aidx] = sp.expand(
            -w*Kgrad_PSI[i][m][n]
        )

# scalar momentum
pflow0[6] = sp.expand(-Epsi_s)
for i in range(3):
    pflow1[i][6] = sp.expand(Psi_g[i])

# vector momenta
for j in range(3):
    pflow0[7+j] = sp.expand(-Epsi_v[j])
    for i in range(3):
        pflow1[i][7+j] = sp.expand(Psi_q[i,j])

traceHpsi = sp.trace(Hpsi)

Pflow0 = []
Pflow1 = [[sp.Integer(0) for _ in range(10)] for __ in range(3)]

for Aidx in range(10):
    scale = 2 if Aidx < 6 else 1
    Pflow0.append(
        sp.expand(
            scale*pflow0[Aidx]
            -sp.Rational(1,2)*P[Aidx]*traceHpsi
        )
    )
    for i in range(3):
        Pflow1[i][Aidx] = sp.expand(
            scale*pflow1[i][Aidx]
        )

P_PSI_FLOW_EXPLICIT = True
print("P_PSI_FLOW_EXPLICIT =",P_PSI_FLOW_EXPLICIT)


P_PSI_FLOW_EXPLICIT = True



# 18 — Flots implicites de \(X\) et \(Y\)

À partir de :

\[
QX=P-J_0,
\qquad
QY=r,
\]

on obtient :

\[
Q\,\delta X
=
\delta P-\delta J_0-(\delta Q)X,
\]

\[
Q\,\delta Y
=
\delta r-(\delta Q)Y.
\]

On garde l'inversion compacte : les coefficients de flot sont définis
comme solutions uniques de systèmes linéaires avec le même \(Q\).

Aucune nouvelle inversion symbolique géante n'est introduite.


In [ ]:

metric_flow_subs = {
    e11:Hpsi[0,0], e22:Hpsi[1,1], e33:Hpsi[2,2],
    e12:Hpsi[0,1], e13:Hpsi[0,2], e23:Hpsi[1,2],
}

dQ_metric_psi = sp.Matrix(dQ).subs(metric_flow_subs)
dJ_metric_psi = sp.Matrix(dJ).subs(metric_flow_subs)
dr_metric_psi = sp.Matrix(dr).subs(metric_flow_subs)

Qflow0 = sp.Matrix(Q.diff(s))*Spsi
for j in range(3):
    Qflow0 += sp.Matrix(Q.diff(v[j]))*Vpsi[j]
Qflow0 += dQ_metric_psi
Qflow0 = sp.Matrix(Qflow0)

Jflow0 = sp.Matrix(J0.diff(s))*Spsi
for j in range(3):
    Jflow0 += sp.Matrix(J0.diff(v[j]))*Vpsi[j]
    Jflow0 += sp.Matrix(J0.diff(g[j]))*gflow0[j]
for i in range(3):
    for j in range(3):
        Jflow0 += sp.Matrix(J0.diff(q[i,j]))*qflow0[i][j]
Jflow0 += dJ_metric_psi

Jflow1 = [sp.zeros(10,1) for _ in range(3)]
for l in range(3):
    for j in range(3):
        Jflow1[l] += sp.Matrix(J0.diff(g[j]))*gflow1[l][j]
    for i in range(3):
        for j in range(3):
            Jflow1[l] += sp.Matrix(J0.diff(q[i,j]))*qflow1[l][i][j]

rflow0 = sp.Matrix(r.diff(s))*Spsi
for j in range(3):
    rflow0 += sp.Matrix(r.diff(v[j]))*Vpsi[j]
rflow0 += dr_metric_psi

ZX0 = sp.Matrix(sp.symbols("ZX0_0:10", real=True))
ZX1 = [
    sp.Matrix(sp.symbols(f"ZX1_{i}_0:10", real=True))
    for i in range(3)
]
ZY0 = sp.Matrix(sp.symbols("ZY0_0:10", real=True))

ZX0_residual = sp.Matrix(
    Q*ZX0
    -(
        sp.Matrix(Pflow0)
        -Jflow0
        -Qflow0*X
    )
)

ZX1_residual = [
    sp.Matrix(
        Q*ZX1[i]
        -(
            sp.Matrix(Pflow1[i])
            -Jflow1[i]
        )
    )
    for i in range(3)
]

ZY0_residual = sp.Matrix(
    Q*ZY0
    -(rflow0-Qflow0*Y)
)

DZX0 = [
    [sp.Symbol(f"DZX0_{i}_{A}") for A in range(10)]
    for i in range(3)
]
DZX1 = [
    [
        [sp.Symbol(f"DZX1_{k}_{i}_{A}") for A in range(10)]
        for i in range(3)
    ]
    for k in range(3)
]

IMPLICIT_XY_PSI_FLOW_SYSTEMS_EXPLICIT = True

print(
    "IMPLICIT_XY_PSI_FLOW_SYSTEMS_EXPLICIT =",
    IMPLICIT_XY_PSI_FLOW_SYSTEMS_EXPLICIT
)


IMPLICIT_XY_PSI_FLOW_SYSTEMS_EXPLICIT = True



# 19 — Correction covariante du jet \(D_kX_A\)

La base cinétique est :

\[
X_A
\sim
(X_{11},X_{22},X_{33},X_{12},X_{13},X_{23},X_s,X_1,X_2,X_3).
\]

Les six premières composantes sont traitées comme un tenseur spatial
symétrique covariant, la septième comme un scalaire et les trois dernières
comme un covecteur spatial.

On ajoute donc explicitement les termes de variation de connexion externe.


In [ ]:

def sym6_to_mat(vec6):
    return sp.Matrix([
        [vec6[0],vec6[3],vec6[4]],
        [vec6[3],vec6[1],vec6[5]],
        [vec6[4],vec6[5],vec6[2]],
    ])

def mat_component_to_A(i,j):
    if i == 0 and j == 0: return 0
    if i == 1 and j == 1: return 1
    if i == 2 and j == 2: return 2
    if {i,j} == {0,1}: return 3
    if {i,j} == {0,2}: return 4
    if {i,j} == {1,2}: return 5
    raise ValueError((i,j))

Xmat = sym6_to_mat(list(X[:6,0]))
Xvec = sp.Matrix([X[7],X[8],X[9]])

DXflow0_derivative_only = [
    [DZX0[k][A] for A in range(10)]
    for k in range(3)
]

DXflow1_derivative_only = [
    [
        [
            sp.expand(
                (ZX0[A] if k==l else 0)
                +DZX1[k][l][A]
            )
            for A in range(10)
        ]
        for k in range(3)
    ]
    for l in range(3)
]

DXflow2 = [
    [
        [
            [
                (ZX1[b][A] if k==a else sp.Integer(0))
                for A in range(10)
            ]
            for k in range(3)
        ]
        for b in range(3)
    ]
    for a in range(3)
]

DXflow0_external = [
    [sp.Integer(0) for A in range(10)]
    for k in range(3)
]

DXflow1_external = [
    [
        [sp.Integer(0) for A in range(10)]
        for k in range(3)
    ]
    for l in range(3)
]

# Tensor sector X_ij.
for k in range(3):
    for i in range(3):
        for j in range(i,3):
            Aidx = mat_component_to_A(i,j)

            DXflow0_external[k][Aidx] = sp.expand(
                -sum(
                    Gamma0[r][k][i]*Xmat[r,j]
                    +Gamma0[r][k][j]*Xmat[i,r]
                    for r in range(3)
                )
            )

            for l in range(3):
                DXflow1_external[l][k][Aidx] = sp.expand(
                    -sum(
                        Gamma1[l][r][k][i]*Xmat[r,j]
                        +Gamma1[l][r][k][j]*Xmat[i,r]
                        for r in range(3)
                    )
                )

# Scalar sector A=6: no external connection correction.

# Covector sector X_i, A=7+i.
for k in range(3):
    for i in range(3):
        Aidx = 7+i

        DXflow0_external[k][Aidx] = sp.expand(
            -sum(
                Gamma0[r][k][i]*Xvec[r]
                for r in range(3)
            )
        )

        for l in range(3):
            DXflow1_external[l][k][Aidx] = sp.expand(
                -sum(
                    Gamma1[l][r][k][i]*Xvec[r]
                    for r in range(3)
                )
            )

DXflow0 = [
    [
        sp.expand(
            DXflow0_derivative_only[k][A]
            +DXflow0_external[k][A]
        )
        for A in range(10)
    ]
    for k in range(3)
]

DXflow1 = [
    [
        [
            sp.expand(
                DXflow1_derivative_only[l][k][A]
                +DXflow1_external[l][k][A]
            )
            for A in range(10)
        ]
        for k in range(3)
    ]
    for l in range(3)
]

assert all(
    sp.expand(
        DXflow0[k][A]
        -DXflow0_derivative_only[k][A]
        -DXflow0_external[k][A]
    ) == 0
    for k in range(3)
    for A in range(10)
)

assert all(
    sp.expand(
        DXflow1[l][k][A]
        -DXflow1_derivative_only[l][k][A]
        -DXflow1_external[l][k][A]
    ) == 0
    for l in range(3)
    for k in range(3)
    for A in range(10)
)

# Scalar component must carry no external connection.
assert all(
    DXflow0_external[k][6] == 0
    for k in range(3)
)
assert all(
    DXflow1_external[l][k][6] == 0
    for l in range(3)
    for k in range(3)
)

COVARIANT_EXTERNAL_CONNECTION_DX_CORRECTED = True
DX_PSI_FLOW_COVARIANTLY_CORRECTED = True

print(
    "COVARIANT_EXTERNAL_CONNECTION_DX_CORRECTED =",
    COVARIANT_EXTERNAL_CONNECTION_DX_CORRECTED
)
print(
    "DX_PSI_FLOW_COVARIANTLY_CORRECTED =",
    DX_PSI_FLOW_COVARIANTLY_CORRECTED
)


COVARIANT_EXTERNAL_CONNECTION_DX_CORRECTED = True
DX_PSI_FLOW_COVARIANTLY_CORRECTED = True



# 20 — Témoins non triviaux des corrections

Un gate ne doit pas passer uniquement parce qu'une correction a été écrite
sous forme symbolique.

On vérifie donc que les blocs externes de connexion ne sont pas
identiquement nuls dans la représentation générale :

\[
\delta\Gamma\neq0
\Longrightarrow
\delta_{\rm ext}(Dg),\
\delta_{\rm ext}(Dq),\
\delta_{\rm ext}(DX)
\not\equiv0
\]

pour au moins un composant générique de chaque secteur.


In [ ]:

DG_EXTERNAL_NONTRIVIAL = any(
    z != 0
    for row in Dgflow0_external
    for z in row
)

DQ_EXTERNAL_NONTRIVIAL = any(
    z != 0
    for slab in Dqflow0_external
    for row in slab
    for z in row
)

DX_EXTERNAL_NONTRIVIAL = any(
    z != 0
    for row in DXflow0_external
    for z in row
)

assert DG_EXTERNAL_NONTRIVIAL
assert DQ_EXTERNAL_NONTRIVIAL
assert DX_EXTERNAL_NONTRIVIAL

print("DG_EXTERNAL_NONTRIVIAL =",DG_EXTERNAL_NONTRIVIAL)
print("DQ_EXTERNAL_NONTRIVIAL =",DQ_EXTERNAL_NONTRIVIAL)
print("DX_EXTERNAL_NONTRIVIAL =",DX_EXTERNAL_NONTRIVIAL)


DG_EXTERNAL_NONTRIVIAL = True
DQ_EXTERNAL_NONTRIVIAL = True
DX_EXTERNAL_NONTRIVIAL = True



# 21 — Réassemblage de \(\mathsf Y_{\rm encoded}\) avec les jets corrigés

On recontracte le vrai \(\rho_{\rm FF}\) avec :

\[
Dg_{\rm cov},
\qquad
Dq_{\rm cov},
\qquad
DX_{\rm cov}.
\]

Le second jet métrique algébrique reste explicitement absent de cette étape.
Ainsi :

\[
\mathsf Y_{\rm encoded,cov}
\]

peut être marqué comme corrigé, mais pas encore comme noyau distributionnel
full-field définitivement fermé.


In [ ]:

Yker0_cov = sp.Integer(0)
Yker1_cov = [sp.Integer(0) for _ in range(3)]
Yker2_cov = [[sp.Integer(0) for _ in range(3)] for __ in range(3)]

# s, v
Yker0_cov += sp.diff(rho_FF,s)*Spsi
for j in range(3):
    Yker0_cov += sp.diff(rho_FF,v[j])*Vpsi[j]

# g
for j in range(3):
    d = sp.diff(rho_FF,g[j])
    Yker0_cov += d*gflow0[j]
    for l in range(3):
        Yker1_cov[l] += d*gflow1[l][j]

# q
for i in range(3):
    for j in range(3):
        d = sp.diff(rho_FF,q[i,j])
        Yker0_cov += d*qflow0[i][j]
        for l in range(3):
            Yker1_cov[l] += d*qflow1[l][i][j]

# P
for Aidx in range(10):
    d = sp.diff(rho_FF,P[Aidx])
    Yker0_cov += d*Pflow0[Aidx]
    for l in range(3):
        Yker1_cov[l] += d*Pflow1[l][Aidx]

# X, Y
for Aidx in range(10):
    dXrho = sp.diff(rho_FF,X[Aidx])
    dYrho = sp.diff(rho_FF,Y[Aidx])

    Yker0_cov += dXrho*ZX0[Aidx]
    Yker0_cov += dYrho*ZY0[Aidx]

    for l in range(3):
        Yker1_cov[l] += dXrho*ZX1[l][Aidx]

# Spatial Einstein tensor
for m in range(3):
    for n in range(m,3):
        d = sp.diff(rho_FF,G3[m,n])

        Yker0_cov += d*dG0[m,n]
        for l in range(3):
            Yker1_cov[l] += d*dG1[l][m,n]
            for r2 in range(3):
                Yker2_cov[l][r2] += d*dG2[l][r2][m,n]

# Dg — corrected
for k in range(3):
    for j in range(3):
        d = sp.diff(rho_FF,Dg[k][j])
        Yker0_cov += d*Dgflow0[k][j]
        for l in range(3):
            Yker1_cov[l] += d*Dgflow1[l][k][j]
            for r2 in range(3):
                Yker2_cov[l][r2] += d*Dgflow2[l][r2][k][j]

# Dq — corrected
for k in range(3):
    for i in range(3):
        for j in range(3):
            d = sp.diff(rho_FF,Dq[k][i][j])
            Yker0_cov += d*Dqflow0[k][i][j]
            for l in range(3):
                Yker1_cov[l] += d*Dqflow1[l][k][i][j]
                for r2 in range(3):
                    Yker2_cov[l][r2] += d*Dqflow2[l][r2][k][i][j]

# DX — corrected
for k in range(3):
    for Aidx in range(10):
        d = sp.diff(rho_FF,DX[k][Aidx])
        Yker0_cov += d*DXflow0[k][Aidx]
        for l in range(3):
            Yker1_cov[l] += d*DXflow1[l][k][Aidx]
            for r2 in range(3):
                Yker2_cov[l][r2] += d*DXflow2[l][r2][k][Aidx]

Y_ENCODED_COVARIANTLY_CORRECTED = True
CONNECTION_EXTERNAL_JETS_CONTRACTED_IN_Y = True

print(
    "Y_ENCODED_COVARIANTLY_CORRECTED =",
    Y_ENCODED_COVARIANTLY_CORRECTED
)
print(
    "CONNECTION_EXTERNAL_JETS_CONTRACTED_IN_Y =",
    CONNECTION_EXTERNAL_JETS_CONTRACTED_IN_Y
)
print("Y encoded covariant order = 2")


Y_ENCODED_COVARIANTLY_CORRECTED = True
CONNECTION_EXTERNAL_JETS_CONTRACTED_IN_Y = True
Y encoded covariant order = 2



# 22 — Différence H8 \(\rightarrow\) correction covariante

On isole la correction apportée à \(\mathsf Y\) :

\[
\Delta\mathsf Y_{\rm cov}
=
\mathsf Y_{\rm encoded,cov}
-
\mathsf Y_{\rm encoded,H8}.
\]

Pour éviter de réutiliser silencieusement un ancien résultat, la différence
est reconstruite directement depuis les blocs externes de connexion.


In [ ]:

Ycov_correction0 = sp.Integer(0)
Ycov_correction1 = [sp.Integer(0) for _ in range(3)]

# Dg correction
for k in range(3):
    for i in range(3):
        d = sp.diff(rho_FF,Dg[k][i])
        Ycov_correction0 += d*Dgflow0_external[k][i]
        for l in range(3):
            Ycov_correction1[l] += d*Dgflow1_external[l][k][i]

# Dq correction
for k in range(3):
    for i in range(3):
        for j in range(3):
            d = sp.diff(rho_FF,Dq[k][i][j])
            Ycov_correction0 += d*Dqflow0_external[k][i][j]
            for l in range(3):
                Ycov_correction1[l] += d*Dqflow1_external[l][k][i][j]

# DX correction
for k in range(3):
    for Aidx in range(10):
        d = sp.diff(rho_FF,DX[k][Aidx])
        Ycov_correction0 += d*DXflow0_external[k][Aidx]
        for l in range(3):
            Ycov_correction1[l] += d*DXflow1_external[l][k][Aidx]

Ycov_correction0 = sp.expand(Ycov_correction0)
Ycov_correction1 = [sp.expand(z) for z in Ycov_correction1]

COVARIANT_Y_CORRECTION_MATERIALIZED = True

print(
    "COVARIANT_Y_CORRECTION_MATERIALIZED =",
    COVARIANT_Y_CORRECTION_MATERIALIZED
)
print(
    "Y covariant correction delta length =",
    len(str(Ycov_correction0))
)
print(
    "Y covariant correction Ddelta lengths =",
    [len(str(z)) for z in Ycov_correction1]
)


COVARIANT_Y_CORRECTION_MATERIALIZED = True
Y covariant correction delta length = 71989
Y covariant correction Ddelta lengths = [18874, 18873, 18873]



# 23 — Gate restant : second jet métrique algébrique

La correction \(Dg,Dq,DX\) ne supprime pas le verrou découvert en H8.

Le noyau \(\rho_{\rm FF}\) contient des premiers jets métriques tels que :

\[
T_C^{mn},
\qquad
T_\psi^{mn},
\qquad
K_{\Gamma,0}^{mn}.
\]

Il reste donc à construire leurs variations sous le flot métrique de
\(\psi\) :

\[
\delta_hT_C^{mn},
\qquad
\delta_hT_\psi^{mn},
\qquad
\delta_hK_{\Gamma,0}^{mn}.
\]

Ce notebook ne les invente pas.


In [ ]:

SECOND_METRIC_ALGEBRAIC_FRECHET_CONTRACTED = False

COVARIANT_DG_DQ_DX_CORRECTION_CLOSED = (
    COVARIANT_EXTERNAL_CONNECTION_DG_CORRECTED
    and COVARIANT_EXTERNAL_CONNECTION_DQ_CORRECTED
    and COVARIANT_EXTERNAL_CONNECTION_DX_CORRECTED
    and CONNECTION_EXTERNAL_JETS_CONTRACTED_IN_Y
    and COVARIANT_Y_CORRECTION_MATERIALIZED
)

Y_DISTRIBUTIONAL_KERNEL_MATERIALIZED = (
    Y_ENCODED_COVARIANTLY_CORRECTED
    and SECOND_METRIC_ALGEBRAIC_FRECHET_CONTRACTED
)

Y_ADJOINT_STRUCTURAL_FORM_EXPLICIT = True
Y_ADJOINT_FULLY_MATERIALIZED = Y_DISTRIBUTIONAL_KERNEL_MATERIALIZED

DISTRIBUTIONAL_DIRAC_INVERSE_FULLY_MATERIALIZED = (
    X_DISTRIBUTIONAL_KERNEL_MATERIALIZED
    and X_ADJOINT_FORMALIZED
    and Y_DISTRIBUTIONAL_KERNEL_MATERIALIZED
    and Y_ADJOINT_FULLY_MATERIALIZED
)

RHH_PHYSICAL_CLASSIFIED = False
HYPERSURFACE_ALGEBRA_GLOBALLY_CLOSED = False
DISPERSION_READY = False

assert COVARIANT_DG_DQ_DX_CORRECTION_CLOSED
assert not SECOND_METRIC_ALGEBRAIC_FRECHET_CONTRACTED
assert not Y_DISTRIBUTIONAL_KERNEL_MATERIALIZED
assert not DISTRIBUTIONAL_DIRAC_INVERSE_FULLY_MATERIALIZED
assert not RHH_PHYSICAL_CLASSIFIED
assert not DISPERSION_READY

print(
    "COVARIANT_DG_DQ_DX_CORRECTION_CLOSED =",
    COVARIANT_DG_DQ_DX_CORRECTION_CLOSED
)
print(
    "SECOND_METRIC_ALGEBRAIC_FRECHET_CONTRACTED =",
    SECOND_METRIC_ALGEBRAIC_FRECHET_CONTRACTED
)
print(
    "Y_DISTRIBUTIONAL_KERNEL_MATERIALIZED =",
    Y_DISTRIBUTIONAL_KERNEL_MATERIALIZED
)
print(
    "DISTRIBUTIONAL_DIRAC_INVERSE_FULLY_MATERIALIZED =",
    DISTRIBUTIONAL_DIRAC_INVERSE_FULLY_MATERIALIZED
)
print("RHH_PHYSICAL_CLASSIFIED =",RHH_PHYSICAL_CLASSIFIED)
print("DISPERSION_READY =",DISPERSION_READY)


COVARIANT_DG_DQ_DX_CORRECTION_CLOSED = True
SECOND_METRIC_ALGEBRAIC_FRECHET_CONTRACTED = False
Y_DISTRIBUTIONAL_KERNEL_MATERIALIZED = False
DISTRIBUTIONAL_DIRAC_INVERSE_FULLY_MATERIALIZED = False
RHH_PHYSICAL_CLASSIFIED = False
DISPERSION_READY = False



# 24 — Verdict strict

Si tous les tests de correction covariante passent, le seul verrou
**explicitement autorisé ensuite** est le second jet métrique algébrique.

Cela ne signifie pas encore que la fermeture Dirac est démontrée.

Le statut doit rester :

\[
\boxed{
R_{HH}^{\rm can}=0
\quad
\texttt{STRONG\_ZERO}
}
\]

et :

\[
\boxed{
R_{HH}^{D}\approx0
\quad
\texttt{CONDITIONAL/STRUCTURAL}.
}
\]


In [ ]:

GATES = {
    "covariant_external_connection_Dg_corrected":
        COVARIANT_EXTERNAL_CONNECTION_DG_CORRECTED,

    "covariant_external_connection_Dq_corrected":
        COVARIANT_EXTERNAL_CONNECTION_DQ_CORRECTED,

    "covariant_external_connection_DX_corrected":
        COVARIANT_EXTERNAL_CONNECTION_DX_CORRECTED,

    "Dg_external_nontrivial":
        DG_EXTERNAL_NONTRIVIAL,

    "Dq_external_nontrivial":
        DQ_EXTERNAL_NONTRIVIAL,

    "DX_external_nontrivial":
        DX_EXTERNAL_NONTRIVIAL,

    "Y_encoded_covariantly_corrected":
        Y_ENCODED_COVARIANTLY_CORRECTED,

    "connection_external_jets_contracted_in_Y":
        CONNECTION_EXTERNAL_JETS_CONTRACTED_IN_Y,

    "covariant_Y_correction_materialized":
        COVARIANT_Y_CORRECTION_MATERIALIZED,

    "covariant_Dg_Dq_DX_correction_closed":
        COVARIANT_DG_DQ_DX_CORRECTION_CLOSED,

    "curvature_Frechet_second_variation_contracted":
        CURVATURE_FRECHET_SECOND_VARIATION_CONTRACTED,

    "second_metric_algebraic_Frechet_contracted":
        SECOND_METRIC_ALGEBRAIC_FRECHET_CONTRACTED,

    "Y_distributional_kernel_materialized":
        Y_DISTRIBUTIONAL_KERNEL_MATERIALIZED,

    "Y_adjoint_fully_materialized":
        Y_ADJOINT_FULLY_MATERIALIZED,

    "distributional_Dirac_inverse_fully_materialized":
        DISTRIBUTIONAL_DIRAC_INVERSE_FULLY_MATERIALIZED,

    "RHH_physical_classified":
        RHH_PHYSICAL_CLASSIFIED,

    "dispersion_ready":
        DISPERSION_READY,
}

for k,vv in GATES.items():
    print(k,":",vv)

if (
    COVARIANT_DG_DQ_DX_CORRECTION_CLOSED
    and not SECOND_METRIC_ALGEBRAIC_FRECHET_CONTRACTED
):
    FINAL_STATUS = (
        "PASS-COVARIANT-DG-DQ-DX-EXTERNAL-CONNECTION-JET-CORRECTION_"
        "BLOCKED-SECOND-METRIC-ALGEBRAIC-FRECHET-JET"
    )
else:
    FINAL_STATUS = (
        "BLOCKED-COVARIANT-DG-DQ-DX-JET-CORRECTION"
    )

R_HH_OPERATIONAL_STATUS = (
    "CANONICAL-STRONG-ZERO_"
    "DIRAC-CONDITIONAL-STRUCTURAL"
)

NEXT_STATUS = (
    "ONLY-SECOND-METRIC-ALGEBRAIC-FRECHET-"
    "CONTRACTION-IN-Y-AUTHORIZED"
)

print("FINAL_STATUS =",FINAL_STATUS)
print("R_HH_OPERATIONAL_STATUS =",R_HH_OPERATIONAL_STATUS)
print("NEXT_STATUS =",NEXT_STATUS)
print("DISPERSION_READY =",DISPERSION_READY)


covariant_external_connection_Dg_corrected : True
covariant_external_connection_Dq_corrected : True
covariant_external_connection_DX_corrected : True
Dg_external_nontrivial : True
Dq_external_nontrivial : True
DX_external_nontrivial : True
Y_encoded_covariantly_corrected : True
connection_external_jets_contracted_in_Y : True
covariant_Y_correction_materialized : True
covariant_Dg_Dq_DX_correction_closed : True
curvature_Frechet_second_variation_contracted : True
second_metric_algebraic_Frechet_contracted : False
Y_distributional_kernel_materialized : False
Y_adjoint_fully_materialized : False
distributional_Dirac_inverse_fully_materialized : False
RHH_physical_classified : False
dispersion_ready : False
FINAL_STATUS = PASS-COVARIANT-DG-DQ-DX-EXTERNAL-CONNECTION-JET-CORRECTION_BLOCKED-SECOND-METRIC-ALGEBRAIC-FRECHET-JET
R_HH_OPERATIONAL_STATUS = CANONICAL-STRONG-ZERO_DIRAC-CONDITIONAL-STRUCTURAL
NEXT_STATUS = ONLY-SECOND-METRIC-ALGEBRAIC-FRECHET-CONTRACTION-IN-Y-AUTHORIZED
DISPERSION_RE


# 25 — Export machine-readable


In [ ]:

artifact = {
    "notebook":
        "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.1.6.2.1.1.1.1",

    "predecessor_executed_sha256":
        UPSTREAM_162111_EXECUTED_SHA256,

    "frozen_input":
        {
            "RHH_canonical":"STRONG_ZERO",
            "RHH_Dirac":"CONDITIONAL_STRUCTURAL",
        },

    "covariant_jet_corrections":
        {
            "Dg":
                COVARIANT_EXTERNAL_CONNECTION_DG_CORRECTED,

            "Dq":
                COVARIANT_EXTERNAL_CONNECTION_DQ_CORRECTED,

            "DX":
                COVARIANT_EXTERNAL_CONNECTION_DX_CORRECTED,

            "contracted_in_Y":
                CONNECTION_EXTERNAL_JETS_CONTRACTED_IN_Y,

            "Y_correction_materialized":
                COVARIANT_Y_CORRECTION_MATERIALIZED,
        },

    "remaining_blocker":
        {
            "second_metric_algebraic_Frechet":
                SECOND_METRIC_ALGEBRAIC_FRECHET_CONTRACTED,

            "Y_full":
                Y_DISTRIBUTIONAL_KERNEL_MATERIALIZED,

            "Dirac_inverse_full":
                DISTRIBUTIONAL_DIRAC_INVERSE_FULLY_MATERIALIZED,
        },

    "gates":GATES,
    "final_status":FINAL_STATUS,
    "R_HH_operational_status":R_HH_OPERATIONAL_STATUS,
    "next_status":NEXT_STATUS,
    "dispersion_ready":False,
}

export_dir = (
    Path("/content/gvh_exports")
    if Path("/content").exists()
    else Path.cwd()/"gvh_exports"
)
export_dir.mkdir(parents=True,exist_ok=True)

artifact_path = export_dir / (
    "gvh_0.3.2.7.3.7.3.3.1.6.2.1.1.1.1_"
    "covariant_Dg_Dq_DX_connection_correction.json"
)

artifact_path.write_text(
    json.dumps(artifact,indent=2),
    encoding="utf-8"
)

print("Artifact:",artifact_path)


Artifact: /content/gvh_exports/gvh_0.3.2.7.3.7.3.3.1.6.2.1.1.1.1_covariant_Dg_Dq_DX_connection_correction.json



# Conclusion

Cette sous-étape cible uniquement la réserve d'audit apparue après H8.

Elle ne modifie ni le résultat canonique ni la classification prudente
du Dirac.

Si l'exécution confirme les asserts :

\[
\boxed{
\delta(D_kg_i)
}
\]

\[
\boxed{
\delta(D_kq_{ij})
}
\]

et :

\[
\boxed{
\delta(D_kX_A)
}
\]

incluent désormais leurs termes externes de connexion.

Le prochain calcul devient alors effectivement :

\[
\boxed{
\text{second jet métrique algébrique de }
T_C,\ T_\psi,\ K_{\Gamma,0}.
}
\]

Mais jusqu'à sa matérialisation :

\[
\boxed{
R_{HH}^{D}\approx0
\quad
\texttt{CONDITIONAL/STRUCTURAL}
}
\]

et :

\[
\boxed{
\texttt{RHH\_PHYSICAL\_CLASSIFIED=False},
\qquad
\texttt{DISPERSION\_READY=False}.
}
\]
